# Reviewer: reproduce PDB D-residue errors in under 5 minutes

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tommaso-R-Marena/ChiralFold/blob/master/demos/Reviewer_5min_Reproduce.ipynb)

This notebook recomputes the **12,573-residue / 29-error** PDB survey from the frozen CSV using **numpy only** (no ChiralFold package, no RCSB network after clone).

**Expected runtime:** ~30–90 seconds on Colab (mostly clone + pip).

**What you will verify**
1. Signed Cα tetrahedron volumes match the frozen survey
2. Exactly **29** D-label/L-coordinate mismatches in **16** structures
3. Optional: clashscore is unchanged after L↔D mirror (isometry)

In [ ]:
# Cell 1 — Clone repo + install numpy (and chiralfold for optional clash demo)
import os, sys, subprocess
if not os.path.isdir('results'):
    !git clone --depth 1 https://github.com/Tommaso-R-Marena/ChiralFold.git
    %cd ChiralFold
!pip -q install numpy
print('cwd:', os.getcwd())
print('csv exists:', os.path.isfile('results/d_residue_verification.csv'))

In [ ]:
# Cell 2 — Recompute all signed volumes from the frozen CSV (~1 s)
!python benchmarks/reproduce_d_residue_errors.py

In [ ]:
# Cell 3 — Show the 16 error structures and taxonomy
import json, pandas as pd
from IPython.display import display, Markdown

summary = json.load(open('results/d_residue_verification_summary.json'))
errors = pd.DataFrame(summary['errors'])
display(Markdown(f"""### Survey headline
- **Checkable residues:** {summary['checkable_residues']:,}
- **Errors:** {summary['l_error']} in **{len(summary['errors_by_structure'])}** structures
- **Rate:** {summary['error_rate_pct']}%
- MolProbity does **not** flag these (L-only Cα check).
"""))
display(errors[['pdb_id','chain','resnum','resname','signed_volume']].sort_values('pdb_id'))

tax = json.load(open('results/error_classification.json'))['classification']
rows = [{'category': k, 'structures': v['structures'], 'errors': v['errors'], 'description': v['description']} for k,v in tax.items()]
display(pd.DataFrame(rows))

In [ ]:
# Cell 4 (optional) — Mirror isometry: clashscore unchanged
# Skips cleanly if chiralfold / rdkit is not installed.
try:
    !pip -q install "chiralfold @ git+https://github.com/Tommaso-R-Marena/ChiralFold.git"
    from chiralfold import audit_pdb, mirror_pdb
    import tempfile, shutil, os
    src = 'chiralfold/data/examples/toy_ubiquitin_fragment.pdb'
    before = audit_pdb(src)['clashes']
    out = tempfile.mktemp(suffix='_mirror.pdb')
    mirror_pdb(src, out, axis='x', rename_residues=True)
    after = audit_pdb(out)['clashes']
    print('Clashscore before:', before['clash_score'], 'n_clashes:', before['n_clashes'])
    print('Clashscore after: ', after['clash_score'], 'n_clashes:', after['n_clashes'])
    assert before['n_clashes'] == after['n_clashes']
    assert abs(before['clash_score'] - after['clash_score']) < 1e-9
    print('PASS — mirror does not introduce steric clashes (isometry).')
    os.remove(out)
except Exception as e:
    print('Optional clash demo skipped or failed:', e)

## Done

You have independently recomputed the survey that underlies the manuscript D-residue findings.

- Full experimental RCSB/CCD cross-check: [`D_Residue_Experimental_Validation.ipynb`](D_Residue_Experimental_Validation.ipynb)
- Unusual cases (macrocycles, CCD ligands, 5M2K): [`Unusual_Cases_and_Clash_Safety.ipynb`](Unusual_Cases_and_Clash_Safety.ipynb)
- Interactive dashboard: [`ChiralFold_Results_Dashboard.ipynb`](ChiralFold_Results_Dashboard.ipynb)